# **CellTracksColab - Track clustering analysis**
---
Explore Spatial Clustering in Track Data with CellTracksColab: This Colab Notebook is designed to help you determine whether tracks exhibit spatial clustering. Before beginning, ensure that your data is properly loaded in the CellTracksColab format for optimal analysis.

---

# **Before getting started**
---

<font size = 5>**Important notes**

---

<font size = 5>**Data Requirements for Analysis**

Be advised of one significant limitation inherent to this notebook.

<font size = 4 color="red">**This notebook only supports 2D + t datasets**</font>.

---
<font size = 5>**Prerequisites for Using This Notebook**

To effectively utilize this notebook the following prerequisite is essential:

1. **DataFrames from CellTrackColab**:
   - Ensure you have the `spots` and `tracks` DataFrames compiled by CellTrackColab.


In [ ]:
# @markdown #MIT License

print("""
**MIT License**

Copyright (c) 2023 Guillaume Jacquemet

Permission is hereby granted, free of charge, to any person obtaining a copy
of this software and associated documentation files (the "Software"), to deal
in the Software without restriction, including without limitation the rights
to use, copy, modify, merge, publish, distribute, sublicense, and/or sell
copies of the Software, and to permit persons to whom the Software is
furnished to do so, subject to the following conditions:

The above copyright notice and this permission notice shall be included in all
copies or substantial portions of the Software.

THE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR
IMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY,
FITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE
AUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER
LIABILITY, WHETHER IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING FROM,
OUT OF OR IN CONNECTION WITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN THE
SOFTWARE.""")

--------------------------------------------------------
# **Part 1. Prepare the session and load the data**
--------------------------------------------------------

## **1.1 Load key dependencies**
---


In [ ]:
#@markdown ##Play to load the dependancies

import sys

# Current version of the notebook the user is running
current_version = "1.1.0"
notebook_name = 'CellTracksColab_Track_Clustering'

# ---------------------------------------------------------
# 1. Environment Detection & Installation (Colab Only)
# ---------------------------------------------------------

user = 'CellMigrationLab'  # Replace with your GitHub username
repo = 'CellTracksColab'

if 'google.colab' in sys.modules:
    print("🚀 Detected Google Colab. Starting installation...")

    # 1. Clone the repository from GitHub

    # We use the PEP 508 syntax: "package[extras] @ url"
    # !pip install -q "nucleisky[all] @ git+https://{token}@github.com/{user}/{repo}.git"
    !git clone https://github.com/{user}/{repo}.git

    sys.path.append("../")
    sys.path.append(f"{repo}/")

    # 2. Install other dependencies
    %pip -q install pandas scikit-learn
    %pip -q install plotly
    %pip -q install tqdm

    # 3. Mount Google Drive
    from google.colab import drive
    drive.mount('/content/gdrive')

    print("✅ Colab setup done")
else:
    sys.path.append("../")

    # Fallback for local environments
    print("✅ Local environment detected. Assuming dependencies are already installed.")

# First of all check if ipywidgets is installed, if not through an informative error
try:
    import ipywidgets as widgets
except ImportError:
    raise ImportError("ipywidgets is not installed. Please install it using 'pip install ipywidgets' or 'conda install -c conda-forge ipywidgets'.")

import matplotlib.pyplot as plt
import ipywidgets as widgets
import seaborn as sns
import pandas as pd
import pandas as pd
import numpy as np
import requests
import os

from IPython.display import display
from ipywidgets import interact
from tqdm.notebook import tqdm

import celltracks
from celltracks import *
from celltracks.Track_Plots import *
from celltracks.BoxPlots_Statistics import *
from celltracks.Track_Metrics import *
from celltracks.Distance_to_ROI import *
from celltracks.Track_Clustering import *

try:
    output_area = widgets.HTML()
        
    # Online version checking file path
    version_file_path = "notebooks/notebook_latest_versions.yaml"
    version_url = f"https://api.github.com/repos/{user}/{repo}/contents/{version_file_path}"

    headers = {"Accept": "application/vnd.github.v3+json"}
    version_response = requests.get(version_url, headers=headers)

    # Check the response status
    if version_response.status_code == 200:
        content = version_response.json()['content']
        decoded_content = base64.b64decode(content).decode('utf-8')
        config = yaml.safe_load(decoded_content)
        latest_version = config.get(notebook_name, "")

        output_area.value = (f"<b>Notebook version:</b> `{current_version}`<br>"
                            f"<b>Latest version available:</b> `{latest_version}`<br>")

        if latest_version == "":
            output_area.value += "⚠️ This notebook is not listed in the version file.<br>"
        elif current_version == latest_version:
            output_area.value += "✅ This notebook is up-to-date.<br>"
        else:
            output_area.value += f"⚠️ A new version of this notebook is available."
    else:
        output_area.value += "⚠️ Could not retrieve the version file.<br>"

    display(output_area)
except ImportError:
    display(widgets.HTML('To check the notebook version, please go to the <a href="../Welcome.ipynb" target="_blank" style="color: #0066cc; text-decoration: underline;">Welcome notebook</a>.'))

print("✅ Environment Ready")

## **1.2. Load existing CellTracksColab dataframes**
---

 Please ensure that your data was properly processed using CellTracksColab. To use the Viewer Notebook, your data must be formatted in the CellTracksColab format. This involves compiling your tracking data into two main DataFrames:

*   Your Track_table: `merged_tracks_df`

*   Spot_table: `merged_spots_df`.

**Data_Dims**: Choose "2D" or "3D" for your data dimensions.

**Results_Folder**: The directory path where the analysis results will be saved.

In [ ]:
#@markdown ##Run this cell to display the UI for loading your dataset

# -----------------------------
# Widget UI
# -----------------------------

Data_Dims_widget = widgets.Dropdown(
    options=["2D", "3D"],
    value="2D",
    description="Data dims:",
    style={"description_width": "120px"}
)

# This was a fixed variable in your original snippet
Data_Type = "CellTracksColab"
Use_test_dataset = False

Track_table_widget = widgets.Text(
    value="",
    placeholder="Path to track table",
    description="Track table:",
    layout=widgets.Layout(width="80%"),
    style={"description_width": "120px"}
)

Spot_table_widget = widgets.Text(
    value="",
    placeholder="Path to spot table",
    description="Spot table:",
    layout=widgets.Layout(width="80%"),
    style={"description_width": "120px"}
)

Results_Folder_widget = widgets.Text(
    value="",
    placeholder="Path to results folder",
    description="Results folder:",
    layout=widgets.Layout(width="80%"),
    style={"description_width": "120px"}
)

submit_button = widgets.Button(
    description="Submit",
    button_style="success",
    tooltip="Load tracking data",
    icon="check"
)

output = widgets.HTML()
process_output = widgets.Output()

# -----------------------------
# Submit button logic
# -----------------------------

def on_submit_clicked(button):
    global Data_Dims
    global Data_Type
    global Track_table
    global Spot_table
    global Use_test_dataset
    global Results_Folder
    global CellTracks
    global merged_spots_df
    global merged_tracks_df
    global File_Format

    output.value = "🔁 Loading data ..."
    process_output.clear_output(wait=True)

    # Read widget values
    Data_Dims = Data_Dims_widget.value
    Track_table = Track_table_widget.value
    Spot_table = Spot_table_widget.value
    Results_Folder = Results_Folder_widget.value
    Use_test_dataset = False

    if Results_Folder != "" and not os.path.exists(Results_Folder):
        os.makedirs(Results_Folder, exist_ok=True)

    with process_output:
        # Update the parameters to load the data
        CellTracks = celltracks.TrackingData()

        if Use_test_dataset:
            # Download the test dataset
            test_celltrackscolab = "https://zenodo.org/record/8420011/files/T_Cells_spots_only.zip?download=1"
            CellTracks.DownloadTestData(test_celltrackscolab)
            File_Format = "csv"
        else:
            CellTracks.Spot_table = Spot_table
            CellTracks.Track_table = Track_table

        CellTracks.Results_Folder = Results_Folder
        CellTracks.skiprows = None
        CellTracks.data_type = Data_Type
        CellTracks.data_dims = Data_Dims

        # Load data
        CellTracks.LoadTrackingData()

        merged_spots_df = CellTracks.spots_data
        check_for_nans(merged_spots_df, "merged_spots_df")
        merged_tracks_df = CellTracks.tracks_data

    output.value = (
        "<br>✅ Done"
        f"<br><br><strong>Results folder:</strong> {Results_Folder}"
        f"<br><strong>Data dimensions:</strong> {Data_Dims}"
        f"<br><strong>Data type:</strong> {Data_Type}"
        f"<br><strong>Track table:</strong> {Track_table}"
        f"<br><strong>Spot table:</strong> {Spot_table}"
    )

submit_button.on_click(on_submit_clicked)

display(
    widgets.HTML("<h2>Provide the path to your CellTracksColab dataset</h2>"),
    Data_Dims_widget,
    widgets.HTML("<h3>Provide the paths to your CellTracksColab tables:</h3>"),
    Track_table_widget,
    Spot_table_widget,
    widgets.HTML("<h3>Provide the path to your Result folder</h3>"),
    Results_Folder_widget,
    submit_button,
    process_output,
    output,
)


## **1.3. Visualise your tracks**
---

In [ ]:
# @markdown ##Run the cell and choose the file you want to inspect
display_plots=True

if not os.path.exists(Results_Folder+"/Tracks"):
    os.makedirs(Results_Folder+"/Tracks")

filenames = merged_spots_df['File_name'].unique()

filename_dropdown = widgets.Dropdown(
    options=filenames,
    value=filenames[0] if len(filenames) > 0 else None,  # Default selected value
    description='File Name:',
)

interact(lambda filename: plot_track_coordinates(filename, merged_spots_df, Results_Folder, display_plots=display_plots), filename=filename_dropdown)


# **Part 2: Assess spatial clustering using Ripley's L function**

In the specific spatial analysis being performed here, the choice of a single point within each track serves to focus on key moments or characteristics of object movement that are particularly relevant to the research objectives. For instance, when analyzing spatial distribution patterns of tracked objects within each field of view (FOV), different analysis points such as the beginning, end, middle, average, or median point of each track offer unique insights. Selecting the "beginning" point might help identify where objects enter an area, while the "end" point can indicate exit locations. Choosing the "middle" point provides insights into where objects spend a significant portion of their time. On the other hand, the "average" or "median" point offers a summary of the overall movement tendencies within each track. By accommodating these various analysis point options, researchers can tailor their spatial analysis to uncover specific aspects of object distribution that are most pertinent to their research questions, enhancing the depth and relevance of their findings.






## **2.1. Choose the point to use for each track**

This section offers users an interactive visualization tool to compare and select the most suitable analysis point within each track for spatial analysis. By providing a dynamic interface, users can assess the impact of different analysis points (e.g., "beginning," "end," "middle," etc.) on spatial distribution patterns. This hands-on exploration empowers users to make informed decisions, ensuring that the chosen analysis point effectively captures the spatial characteristics of each track. Ultimately, this customization enhances the precision and relevance of spatial analysis results for a wide range of research objectives.







In [ ]:
# @markdown ##Run the cell and choose the analysis point you want to use

base_folder = f"{Results_Folder}/Track_Clustering/Tracks"

# Check and create necessary directories
if not os.path.exists(base_folder):
    os.makedirs(base_folder)

# Extract unique filenames from the dataframe
filenames = merged_spots_df['File_name'].unique()

# Create Dropdown widgets with labels and fixed width
filename_dropdown = widgets.Dropdown(
    options=filenames,
    value=filenames[0] if len(filenames) > 0 else None,  # Default selected value
    description='File Name:',
    layout=widgets.Layout(width='300px'),  # Adjust width as needed
)

analysis_option_dropdown = widgets.Dropdown(
    options=["beginning", "end", "middle", "average", "median"],
    value="beginning",
    description='Point:',
    layout=widgets.Layout(width='300px'),  # Adjust width as needed
)

# Link both Dropdown widgets to the plotting function
interact(lambda filename, analysis_option: plot_coordinates_Clustering(filename, merged_spots_df, analysis_option, base_folder),
         filename=filename_dropdown, analysis_option=analysis_option_dropdown);

## **2.2. Compute Ripley's L function for each FOV**

This code aims to compute Ripley's L function for each Field of View (FOV) in a dataset of tracked objects. Ripley's L function is a spatial statistics tool used to analyze the spatial distribution of points or objects in a given area. In this analysis, we are interested in understanding how objects are distributed within each FOV.

**User Input Options**

1. **Analysis Option**: This option allows you to choose the point within each track that will be used for analysis. You can select one of the following options:
   - "beginning": Use the initial position of each track.
   - "end": Use the final position of each track.
   - "middle": Use the middle position of each track.
   - "average": Use the average position of all points within each track.
   - "median": Use the median position of all points within each track.

2. **r_values Range**: Ripley's L function is computed for a range of spatial distances denoted by "r." You can specify the range of r_values using the following parameters:
   - **Start Value**: The starting value of "r" (minimum distance). This number should be greater than 0.
   - **End Value**: The ending value of "r" (maximum distance).
   - **Number of Points**: The number of points or steps within the specified range. The analysis will be performed at equidistant intervals between the start and end values.



In [ ]:
#@markdown ##Run this cell to check the settings you want to use to compute Ripley's L function

# -----------------------------
# Widget UI
# -----------------------------

analysis_option_widget = widgets.Dropdown(
    options=["beginning", "end", "middle", "average", "median"],
    value="middle",
    description="Point:",
    style={"description_width": "120px"}
)

r_values_start_widget = widgets.FloatText(
    value=0.1,
    description="R start:",
    style={"description_width": "120px"}
)

r_values_end_widget = widgets.FloatText(
    value=100,
    description="R end:",
    style={"description_width": "120px"}
)

r_values_count_widget = widgets.IntText(
    value=50,
    description="R count:",
    style={"description_width": "120px"}
)

submit_button = widgets.Button(
    description="Submit",
    button_style="success",
    tooltip="Check Ripley's L settings",
    icon="check"
)

output = widgets.HTML()
process_output = widgets.Output()

# -----------------------------
# Submit button logic
# -----------------------------

def on_submit_clicked(button):
    global analysis_option
    global r_values_start
    global r_values_end
    global r_values_count
    global r_values
    global max_dims
    global max_x
    global max_y
    global edge_points_count
    global total_points_count
    global fov_count

    output.value = "🔁 Testing the parameters ..."
    process_output.clear_output(wait=True)

    # Read widget values
    analysis_option = analysis_option_widget.value
    r_values_start = r_values_start_widget.value
    r_values_end = r_values_end_widget.value
    r_values_count = r_values_count_widget.value

    with process_output:
        r_values = np.linspace(r_values_start, r_values_end, r_values_count)

        # Get the dimensions with fixed minimum values
        max_dims = get_dimensions(merged_spots_df)
        max_x, max_y = max_dims["max_x"], max_dims["max_y"]

        # Check that r_values_end is not larger than the field of view
        if r_values_end > max_x or r_values_end > max_y:
            output.value = "❌ r_values_end is larger than the field of view dimensions."
            return

        # Initialize a counter for edge points
        edge_points_count = 0
        total_points_count = 0
        fov_count = merged_spots_df["File_name"].nunique()

        for file_name, group in tqdm(merged_spots_df.groupby("File_name"), desc="Testing the parameters"):
            # Sort each track by POSITION_T
            group = group.sort_values(by=["TRACK_ID", "POSITION_T"])

            representative_points = group.groupby("TRACK_ID").apply(
                lambda track: select_analysis_point(track, analysis_option)
            ).dropna()

            total_points_count += len(representative_points)

            # Identify points too close to the edge of the image
            edge_points = representative_points[
                (representative_points["POSITION_X"] <= r_values_end) |
                (representative_points["POSITION_X"] >= (max_x - r_values_end)) |
                (representative_points["POSITION_Y"] <= r_values_end) |
                (representative_points["POSITION_Y"] >= (max_y - r_values_end))
            ]

            # Count edge points
            edge_points_count += len(edge_points)

    output.value = (
        "<br>✅ Done"
        f"<br><br><strong>Analysis point:</strong> {analysis_option}"
        f"<br><strong>R values start:</strong> {r_values_start}"
        f"<br><strong>R values end:</strong> {r_values_end}"
        f"<br><strong>R values count:</strong> {r_values_count}"
        f"<br><strong>Total number of points:</strong> {total_points_count}"
        f"<br><strong>Total number of fields of view:</strong> {fov_count}"
        f"<br><strong>Total points closer to the edge than r_values_end:</strong> {edge_points_count}"
        "<br>These points could contribute to edge effects and impact your analysis."
    )

submit_button.on_click(on_submit_clicked)

display(
    widgets.HTML("<h2>Check the settings you want to use to compute Ripley's L function</h2>"),
    analysis_option_widget,
    r_values_start_widget,
    r_values_end_widget,
    r_values_count_widget,
    submit_button,
    process_output,
    output,
)


In [ ]:
# @markdown ##Run to compute Ripley's L function for each FOV

# Check and create necessary directories
results_dir = f"{Results_Folder}/Track_Clustering/RipleyL"
os.makedirs(results_dir, exist_ok=True)

# Define area based on your dataset's extent
area = (max_x - merged_spots_df['POSITION_X'].min()) * (max_y - merged_spots_df['POSITION_Y'].min())

# Compute Ripley's L function for each FOV
l_values_per_fov_slow = {}
for file_name, group in tqdm(merged_spots_df.groupby('File_name'), desc="Processing FOVs"):
    # Sort each track by POSITION_T
    group = group.sort_values(by=['TRACK_ID', 'POSITION_T'])

    # Select representative points for each track
    representative_points = group.groupby('TRACK_ID').apply(lambda track: select_analysis_point(track, analysis_option)).dropna()
    l_values = [ripley_l(representative_points[['POSITION_X', 'POSITION_Y']].values, r, area) for r in r_values]
    l_values_per_fov_slow[file_name] = l_values


## **2.3. Compute Monte Carlo Simulations for Each FOV**

This code section performs Monte Carlo simulations to assess the significance of the observed spatial distribution patterns within each Field of View (FOV) in a dataset of tracked objects. The simulations help establish confidence envelopes for Ripley's L function, allowing for statistical testing.


**Number of Simulations (Nb_simulation)**: You can specify the number of Monte Carlo simulations to run for each FOV. This parameter determines the level of statistical confidence and computational resources used in the analysis.

In [ ]:
#@markdown ##Run this cell to compute Monte Carlo simulations for each FOV

# -----------------------------
# Widget UI
# -----------------------------

Nb_simulation_widget = widgets.IntText(
    value=10,
    description="Simulations:",
    style={"description_width": "120px"}
)

submit_button = widgets.Button(
    description="Submit",
    button_style="success",
    tooltip="Compute Monte Carlo simulations",
    icon="check"
)

output = widgets.HTML()
process_output = widgets.Output()

# -----------------------------
# Submit button logic
# -----------------------------

def on_submit_clicked(button):
    global Nb_simulation
    global simulated_l_values_dict_slow
    global confidence_envelopes_slow
    global area

    output.value = "🔁 Computing Monte Carlo simulations for each FOV ..."
    process_output.clear_output(wait=True)

    # Read widget values
    Nb_simulation = Nb_simulation_widget.value

    simulated_l_values_dict_slow = {}
    confidence_envelopes_slow = {}

    with process_output:
        # Compute the area once
        area = (
            (merged_spots_df["POSITION_X"].max() - merged_spots_df["POSITION_X"].min()) *
            (merged_spots_df["POSITION_Y"].max() - merged_spots_df["POSITION_Y"].min())
        )

        for file_name, group in tqdm(merged_spots_df.groupby("File_name"), desc="Processing FOVs"):

            group = group.sort_values(by=["TRACK_ID", "POSITION_T"])
            representative_points = group.groupby("TRACK_ID").apply(
                lambda track: select_analysis_point(track, analysis_option)
            ).dropna()

            if not representative_points.empty:
                simulations = [
                    simulate_random_points(
                        len(representative_points),
                        (merged_spots_df["POSITION_X"].min(), merged_spots_df["POSITION_X"].max()),
                        (merged_spots_df["POSITION_Y"].min(), merged_spots_df["POSITION_Y"].max())
                    )
                    for _ in tqdm(range(Nb_simulation), desc=f"Simulating for {file_name}", leave=False)
                ]

                simulated_l_values = [
                    [ripley_l(points, r, area) for r in r_values]
                    for points in simulations
                ]
                simulated_l_values_dict_slow[file_name] = simulated_l_values

                lower_bound = np.percentile(simulated_l_values, 2.5, axis=0)
                upper_bound = np.percentile(simulated_l_values, 97.5, axis=0)
                confidence_envelopes_slow[file_name] = (lower_bound, upper_bound)

    output.value = (
        "<br>✅ Monte Carlo simulations completed for all fields of view."
        f"<br><br><strong>Simulations per FOV:</strong> {Nb_simulation}"
        f"<br><strong>Fields of view processed:</strong> {len(simulated_l_values_dict_slow)}"
    )

submit_button.on_click(on_submit_clicked)

display(
    widgets.HTML("<h2>Compute Monte Carlo simulations for each FOV</h2>"),
    Nb_simulation_widget,
    submit_button,
    process_output,
    output,
)


## **2.4. Plots the results for each FOV**


In [ ]:
#@markdown ##Run this cell to plot the results for each FOV

# -----------------------------
# Widget UI
# -----------------------------

show_plots_widget = widgets.Checkbox(
    value=False,
    description="Show plots"
)

submit_button = widgets.Button(
    description="Submit",
    button_style="success",
    tooltip="Plot results for each FOV",
    icon="check"
)

output = widgets.HTML()
process_output = widgets.Output()

# -----------------------------
# Submit button logic
# -----------------------------

def on_submit_clicked(button):
    global show_plots
    global pdf_path

    output.value = "🔁 Plotting the results for each FOV ..."
    process_output.clear_output(wait=True)

    # Read widget values
    show_plots = show_plots_widget.value

    saved_plots = 0
    missing_envelopes = 0

    with process_output:
        # Visualization of Ripley's L function with confidence envelopes
        for file_name, l_values in l_values_per_fov_slow.items():
            # Retrieve the confidence envelope for the current file
            lower_bound, upper_bound = confidence_envelopes_slow.get(file_name, (None, None))

            # Only proceed if the confidence envelope exists
            if lower_bound is not None and upper_bound is not None:
                plt.figure(figsize=(10, 6))
                plt.plot(r_values, l_values, label=f"L(r) for {file_name}")
                plt.fill_between(r_values, lower_bound, upper_bound, color="gray", alpha=0.5)
                plt.xlabel("Radius (r)")
                plt.ylabel("Ripley's L Function")
                plt.title(f"Ripley's L Function - {file_name}_{analysis_option}")
                plt.legend()
                plt.grid(True)

                # Save the plot as a PDF in the specified folder
                pdf_path = os.path.join(
                    Results_Folder,
                    "Track_Clustering",
                    "RipleyL",
                    f"{file_name}_{analysis_option}.pdf"
                )
                plt.savefig(pdf_path, bbox_inches="tight")
                saved_plots += 1

                if show_plots:
                    plt.show()
                    plt.close()
                else:
                    plt.close()
            else:
                print(f"No confidence envelope data available for {file_name}_{analysis_option}")
                missing_envelopes += 1

    output.value = (
        "<br>✅ Done"
        f"<br><br><strong>Plots saved in:</strong> {os.path.join(Results_Folder, 'Track_Clustering', 'RipleyL')}"
        f"<br><strong>Show plots:</strong> {show_plots}"
        f"<br><strong>Plots saved:</strong> {saved_plots}"
        f"<br><strong>Missing confidence envelopes:</strong> {missing_envelopes}"
    )

submit_button.on_click(on_submit_clicked)

display(
    widgets.HTML("<h2>Plot the results for each FOV</h2>"),
    show_plots_widget,
    submit_button,
    process_output,
    output,
)


## **2.5. Chose a specific radius and plot the results**


In [ ]:
#@markdown ##Run this cell to define a specific radius and run

# -----------------------------
# Widget UI
# -----------------------------

specific_radius_widget = widgets.FloatText(
    value=50,
    description="Radius:",
    style={"description_width": "120px"}
)

submit_button = widgets.Button(
    description="Submit",
    button_style="success",
    tooltip="Analyze specific radius",
    icon="check"
)

output = widgets.HTML()
process_output = widgets.Output()

# -----------------------------
# Submit button logic
# -----------------------------

def on_submit_clicked(button):
    global specific_radius
    global specific_radius_index
    global l_values_at_specific_radius_slow
    global rows
    global confidence_df
    global additional_info_df
    global merged_df
    global pdf_path

    output.value = "🔁 Analyzing Ripley's L values at the selected radius ..."
    process_output.clear_output(wait=True)

    # Read widget values
    specific_radius = specific_radius_widget.value

    with process_output:
        os.makedirs(os.path.join(Results_Folder, "Track_Clustering", "RipleyL"), exist_ok=True)

        # Extract L values at the specific radius
        specific_radius_index = np.argmin(np.abs(r_values - specific_radius))
        l_values_at_specific_radius_slow = {
            fov: l_values[specific_radius_index]
            for fov, l_values in l_values_per_fov_slow.items()
        }

        # Plotting
        plt.figure(figsize=(12, 6))
        plt.bar(l_values_at_specific_radius_slow.keys(), l_values_at_specific_radius_slow.values())
        plt.xlabel("Field of View")
        plt.ylabel(f"Ripley's L at radius {specific_radius}")
        plt.title(f"Comparison of Ripley's L Function at Radius {specific_radius} Across Different FOVs")
        plt.xticks(rotation=45)

        # Save the plot as a PDF in the specified folder
        pdf_path = os.path.join(
            Results_Folder,
            "Track_Clustering",
            "RipleyL",
            f"l_values_at_specific_radius_{specific_radius}_{analysis_option}.pdf"
        )
        plt.savefig(pdf_path, bbox_inches="tight")

        plt.show()

        # Create DataFrame with confidence envelopes, median, and L values at the specific radius
        rows = []
        for fov, (lower_bound, upper_bound) in confidence_envelopes_slow.items():
            l_value = l_values_per_fov_slow[fov][specific_radius_index]
            lower = lower_bound[specific_radius_index]
            upper = upper_bound[specific_radius_index]

            # Retrieve simulated L values for the FOV
            simulated_l_values_for_fov_slow = simulated_l_values_dict_slow.get(fov, [])

            # Calculate median if simulated L values are available for the FOV
            if simulated_l_values_for_fov_slow:
                median_vals = [l_vals[specific_radius_index] for l_vals in simulated_l_values_for_fov_slow]
                median = np.median(median_vals) if median_vals else np.nan
            else:
                median = np.nan

            rows.append([fov, l_value, lower, upper, median])

        confidence_df = pd.DataFrame(
            rows,
            columns=["File_name", "Ripley_L_at_Specific_Radius", "Lower_Bound", "Upper_Bound", "Median"]
        )

        # Merge with additional information
        additional_info_df = merged_tracks_df[
            ["File_name", "Condition", "experiment_nb", "Repeat"]
        ].drop_duplicates("File_name")

        merged_df = pd.merge(
            confidence_df,
            additional_info_df,
            left_on="File_name",
            right_on="File_name"
        )

        # Save the merged DataFrame to a CSV file
        csv_path = os.path.join(
            Results_Folder,
            "Track_Clustering",
            "RipleyL",
            f"ripleys_l_values__{specific_radius}_{analysis_option}.csv"
        )
        merged_df.to_csv(csv_path, index=False)

    output.value = (
        "<br>✅ Done"
        f"<br><br><strong>Specific radius:</strong> {specific_radius}"
        f"<br><strong>Closest radius value:</strong> {r_values[specific_radius_index]}"
        f"<br><strong>Saved plot:</strong> {pdf_path}"
        f"<br><strong>Saved table:</strong> {csv_path}"
    )

submit_button.on_click(on_submit_clicked)

display(
    widgets.HTML("<h2>Define a specific radius and run</h2>"),
    specific_radius_widget,
    submit_button,
    process_output,
    output,
)


## **2.6. Comparison of Ripley's L Values Across Conditions**



In [ ]:
# @markdown ##Comparison of Ripley\'s L Values Across Conditions

# Convert 'Condition' to string if it's not already
merged_df['Condition'] = merged_df['Condition'].astype(str)

# Create the box plot
plt.figure(figsize=(12, 8))
sns.boxplot(data=merged_df, x='Condition', y='Ripley_L_at_Specific_Radius')

# Overlay the Monte Carlo simulation results
for condition in merged_df['Condition'].unique():
    condition_data = merged_df[merged_df['Condition'] == condition]

    # Plot median values
    medians = condition_data['Median']
    plt.scatter([condition] * len(medians), medians, color='red', alpha=0.5)  # Median

    # Handle NaN values and calculate mean and error only for non-NaN values
    valid_data = condition_data.dropna(subset=['Median', 'Lower_Bound', 'Upper_Bound'])
    if not valid_data.empty:
        median_mean = valid_data['Median'].mean()
        lower_mean = valid_data['Lower_Bound'].mean()
        upper_mean = valid_data['Upper_Bound'].mean()
        yerr = [[median_mean - lower_mean], [upper_mean - median_mean]]

        # Check if yerr contains valid data before plotting
        if not any(np.isnan(yerr)):
            plt.errorbar(condition, median_mean, yerr=yerr, fmt='o', color='black', alpha=0.5)  # Confidence interval

# Add labels and title
plt.xlabel('Condition')
plt.ylabel(f"Ripley's L at radius {specific_radius}")
plt.title('Comparison of Ripley\'s L Values Across Conditions with Monte Carlo Simulation Results')
plt.xticks(rotation=45)
plt.grid(True)

# Save the figure before showing it
pdf_path = os.path.join(f"{Results_Folder}/Track_Clustering/RipleyL/l_values_Conditions_radius_{specific_radius}_{analysis_option}.pdf")
plt.savefig(pdf_path, bbox_inches='tight')

# Show the plot
plt.show()


## **2.7. Plot the analysis point for each FOV**


In [ ]:
#@markdown ##Run this cell to plot the coordinates used for the spatial analysis for all FOV

# -----------------------------
# Widget UI
# -----------------------------

show_plots_widget = widgets.Checkbox(
    value=False,
    description="Show plots"
)

submit_button = widgets.Button(
    description="Submit",
    button_style="success",
    tooltip="Plot spatial analysis coordinates for all FOV",
    icon="check"
)

output = widgets.HTML()
process_output = widgets.Output()

# -----------------------------
# Submit button logic
# -----------------------------

def on_submit_clicked(button):
    global show_plots
    global filenames
    global analysis_option

    output.value = "🔁 Plotting coordinates used for the spatial analysis for all FOV ..."
    process_output.clear_output(wait=True)

    # Read widget values
    show_plots = show_plots_widget.value

    # Ensure the directory for saving plots exists
    if not os.path.exists(f"{Results_Folder}/Track_Clustering/Coordinates"):
        os.makedirs(f"{Results_Folder}/Track_Clustering/Coordinates")

    filenames = merged_spots_df["File_name"].unique()

    with process_output:
        for filename in filenames:
            analysis_option = "beginning"
            plot_clustering_coordinates(
                filename,
                merged_spots_df,
                analysis_option,
                Results_Folder,
                r_values_end,
                max_x,
                max_y,
                show_plots
            )

    output.value = (
        "<br>✅ Done"
        f"<br><br><strong>Coordinates saved in:</strong> {Results_Folder}/Track_Clustering/Coordinates"
        f"<br><strong>Show plots:</strong> {show_plots}"
        f"<br><strong>Fields of view processed:</strong> {len(filenames)}"
        f"<br><strong>Analysis point:</strong> {analysis_option}"
    )

submit_button.on_click(on_submit_clicked)

display(
    widgets.HTML("<h2>Plot the coordinates used for the spatial analysis for all FOV</h2>"),
    show_plots_widget,
    submit_button,
    process_output,
    output,
)


# **Part 3: Version log**
---
While we strive to provide accurate and helpful information, please be aware that:
  - This notebook may contain bugs.
  - Features are currently limited and will be expanded in future releases.

We encourage users to report any issues or suggestions for improvement.

**Version 1.1.0**
  - Make the notebook locally usable:
    - Replace Google Colab's parameter widgets with ipywidgets
    - Merge the initial Google Colab related code with the imports code cell
    - Remove unnecesary or repeated imports
    - Include additional functionalities for local vs Colab usage
    - Update notebook versioning checking

**Version 1.0.1**
  - Includes a general data reader
  - Plotting functions are imported from the main code
    
**Version 0.9.1**
  - Improved documentation
  - Improved saving strategy

**Version 0.8**
  - First release of this notebook

